In [5]:
# ============================================================================
# WIZUALIZACJE MATEMATYCZNE - Lista 3 (Zadania 1-21)
# ============================================================================
# Kod wykorzystuje słowniki do przechowywania funkcji i parametrów
# Każdy wykres ma tytuł, podpisane osie i legendę
# Pliki automatycznie trafiają do podfolderów i stare pliki są czyszczone
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Polygon, Circle
from matplotlib.backends.backend_pdf import PdfPages
from scipy.interpolate import splprep, splev, interp1d
import sympy as sp
from sympy import symbols, diff, solve, Piecewise, cos, sin, sqrt, exp, ln, atan, pi, simplify
from sympy.plotting import plot as sp_plot, plot3d as sp_plot3d, plot_implicit
import shutil
import os
from pathlib import Path

# Ustawienia matplotlib
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2

# Ścieżka do zapisu
OUTPUT_DIR = Path("wykresy_wyjscie")
OUTPUT_DIR.mkdir(exist_ok=True)

# Podfoldery dla organizacji plików
SUBFOLDERS = {
    'pdf': OUTPUT_DIR / "PDF_indywidualne",
    'png': OUTPUT_DIR / "PNG",
    'jpg': OUTPUT_DIR / "JPG",
    'main_pdf': OUTPUT_DIR / "PDF_do_odesłania"
}

# Tworzenie podfolderów
for folder in SUBFOLDERS.values():
    folder.mkdir(exist_ok=True)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def cleanup_old_files():
    """Czyszczenie starych plików przed nowym uruchomieniem"""
    print("🧹 Czyszczenie starych plików...\n")
    
    # Czyszczenie PNG
    for f in SUBFOLDERS['png'].glob("*.png"):
        f.unlink()
    print(f"   ✓ Usunięto PNG z {SUBFOLDERS['png'].name}/")
    
    # Czyszczenie JPG
    for f in SUBFOLDERS['jpg'].glob("*.jpg"):
        f.unlink()
    print(f"   ✓ Usunięto JPG z {SUBFOLDERS['jpg'].name}/")
    
    # Czyszczenie PDF_indywidualne (ale NIE PDF_do_odesłania)
    for f in SUBFOLDERS['pdf'].glob("*.pdf"):
        f.unlink()
    print(f"   ✓ Usunięto PDF-y z {SUBFOLDERS['pdf'].name}/")
    
    print()

def setup_plot(title, xlabel, ylabel, figsize=(10, 7)):
    """Tworzenie i setup figury z tytułem i podpisami osi"""
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.grid(True, alpha=0.3)
    return fig, ax

def save_figure(fig, filename_base):
    """Zapis figury w formatach PDF, PNG, JPG (bezpośrednio do podfolderów)"""
    pdf_path = SUBFOLDERS['pdf'] / f"{filename_base}.pdf"
    png_path = SUBFOLDERS['png'] / f"{filename_base}.png"
    jpg_path = SUBFOLDERS['jpg'] / f"{filename_base}.jpg"
    
    fig.savefig(pdf_path, bbox_inches='tight', dpi=300)
    fig.savefig(png_path, bbox_inches='tight', dpi=150)
    fig.savefig(jpg_path, bbox_inches='tight', dpi=150)
    plt.close(fig)
    
    print(f"✓ Zapisano: {filename_base}")
    return str(pdf_path), str(png_path), str(jpg_path)

def sympy_to_numpy(expr, var, x_vals):
    """Konwersja wyrażenia SymPy na funkcję NumPy"""
    f = sp.lambdify(var, expr, 'numpy')
    return f(x_vals)

def find_critical_points(f, x_var, a, b):
    """Znalezienie ekstremów lokalnych funkcji w przedziale [a, b]"""
    df = diff(f, x_var)
    critical = solve(df, x_var)
    critical = [float(pt.evalf()) for pt in critical if pt.is_real and a <= float(pt.evalf()) <= b]
    return sorted(critical)

# Wyczyść stare pliki na starcie
cleanup_old_files()

print("✓ Importy i helper functions załadowane")

🧹 Czyszczenie starych plików...

   ✓ Usunięto PNG z PNG/
   ✓ Usunięto JPG z JPG/
   ✓ Usunięto PDF-y z PDF_indywidualne/

✓ Importy i helper functions załadowane


In [ ]:
# ZADANIA 1-8 (WERSJA PODSTAWOWA)
x = symbols('x')
y = symbols('y')

#f(x) = x^4 - 4x^2
f1 = x**4 - 4*x**2
x_vals = np.linspace(-3, 3, 500)
y_vals = sympy_to_numpy(f1, x, x_vals)

# Znalezienie ekstremów
critical_pts = find_critical_points(f1, x, -3, 3)
critical_vals = [float(f1.subs(x, cp).evalf()) for cp in critical_pts]

fig, ax = setup_plot('Zadanie 1: Funkcja z ekstremami lokalnymi',
                      r'$x$', r'$f(x) = x^4 - 4x^2$')
ax.plot(x_vals, y_vals, 'b-', linewidth=2.5, label='f(x)')
ax.scatter(critical_pts, critical_vals, color='red', s=100, zorder=5, label='Ekstrema', marker='o')

# Oznaczenie ekstremów
for cp, cv in zip(critical_pts, critical_vals):
    ax.annotate(f'({cp:.2f}, {cv:.2f})', xy=(cp, cv), xytext=(0, 10), 
                textcoords='offset points', ha='center', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "01_ekstrema")

print("ZADANIE 2: Trzy funkcje z ograniczonymi zakresami")

functions_2 = {
    'f1': {'expr': x**2, 'range': [-10, -1], 'color': 'blue', 'label': r'$f_1(x) = x^2, x \in [-10, -1]$'},
    'f2': {'expr': x**3, 'range': [-5, 3], 'color': 'green', 'label': r'$f_2(x) = x^3, x \in [-5, 3]$'},
    'f3': {'expr': x + 1, 'range': [4, 10], 'color': 'red', 'label': r'$f_3(x) = x + 1, x \in [4, 10]$'}
}

fig, ax = setup_plot('Zadanie 2: Trzy funkcje na jednym wykresie',
                      r'$x$', r'$y$')

for key, func_dict in functions_2.items():
    x_range = func_dict['range']
    x_vals = np.linspace(x_range[0], x_range[1], 200)
    y_vals = sympy_to_numpy(func_dict['expr'], x, x_vals)
    ax.plot(x_vals, y_vals, color=func_dict['color'], linewidth=2.5, label=func_dict['label'])

ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "02_trzy_funkcje")

print("ZADANIE 3: Funkcja klamerkowa i jej pochodna")

# Funkcja klamerkowa: 1 + x^2 dla x > 5, 4 dla x = 5, -3x + 4 dla x < 5
f_piecewise = Piecewise(
    (-3*x + 4, x < 5),
    (4, x == 5),
    (1 + x**2, x > 5)
)

# Pochodna (obliczana dla poszczególnych części)
f_prime_piecewise = Piecewise(
    (-3, x < 5),
    (sp.oo, x == 5),  # Undefined w punkcie przeskoku
    (2*x, x > 5)
)

# Wizualizacja
x_left = np.linspace(0, 5, 200)
x_right = np.linspace(5, 10, 200)

y_left = sympy_to_numpy(-3*x + 4, x, x_left)
y_right = sympy_to_numpy(1 + x**2, x, x_right)

y_prime_left = np.full_like(x_left, -3.0)  # Stała pochodna = -3
y_prime_right = sympy_to_numpy(2*x, x, x_right)

fig, ax = setup_plot('Zadanie 3: Funkcja klamerkowa i jej pochodna',
                      r'$x$', r'$y$')

ax.plot(x_left, y_left, 'b-', linewidth=2.5, label='f(x) dla x < 5: -3x + 4')
ax.plot(x_right, y_right, 'b-', linewidth=2.5, label='f(x) dla x > 5: 1 + x^2')
ax.scatter([5], [4], color='blue', s=100, zorder=5)

ax.plot(x_left, y_prime_left, 'r--', linewidth=2.5, label="f'(x) dla x < 5: -3")
ax.plot(x_right, y_prime_right, 'r--', linewidth=2.5, label="f'(x) dla x > 5: 2x")

ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.axvline(x=5, color='gray', linewidth=1, linestyle=':', alpha=0.5)
ax.legend(loc='best', fontsize=9)
save_figure(fig, "03_klamerkowa")

print("ZADANIE 4: Trójkąt utworzony przez trzy proste")


lines_4 = {
    'l1': {'a': 2, 'b': 0, 'label': r'$y = 2x$'},          # y = 2x
    'l2': {'a': -1, 'b': 6, 'label': r'$y = -x + 6$'},     # y = -x + 6
    'l3': {'vertical': True, 'x_val': 1, 'label': r'$x = 1$'}
}

plt.xline()

vertices = np.array([[2, 4], [1, 2], [1, 5]])

fig, ax = setup_plot('Zadanie 4: Trójkąt z trzema prostymi',
                      r'$x$', r'$y$')

x_range = np.linspace(-0.5, 4, 200)

# Rysowanie prostych
y_l1 = 2 * x_range
y_l2 = -x_range + 6
ax.plot(x_range, y_l1, 'b-', linewidth=2, label=r'$y = 2x$')
ax.plot(x_range, y_l2, 'g-', linewidth=2, label=r'$y = -x + 6$')
ax.axvline(x=1, color='r', linewidth=2, label=r'$x = 1$')

# Zaznaczenie trójkąta
triangle = Polygon(vertices, facecolor='yellow', alpha=0.5, edgecolor='black', linewidth=2)
ax.add_patch(triangle)

# Wierzchołki
ax.scatter(vertices[:, 0], vertices[:, 1], color='red', s=100, zorder=5, marker='o')

ax.set_xlim(-0.5, 4)
ax.set_ylim(-0.5, 7)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "04_trojkat")

print("ZADANIE 5: Krzywa uwikłana (elipsa)")

# Elipsa: x^2/4 + y^2/9 = 1
# Parametryzacja: x = 2*cos(t), y = 3*sin(t)
t_vals = np.linspace(0, 2*np.pi, 500)
x_ellipse = 2 * np.cos(t_vals)
y_ellipse = 3 * np.sin(t_vals)

fig, ax = setup_plot(r'Zadanie 5: Krzywa uwikłana (elipsa) $\frac{x^2}{4} + \frac{y^2}{9} = 1$',
                      r'$x$', r'$y$')

ax.plot(x_ellipse, y_ellipse, 'b-', linewidth=2.5, label='Elipsa')
ax.scatter([0], [0], color='red', s=100, marker='+', linewidths=2, label='Środek')

ax.set_xlim(-3, 3)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
save_figure(fig, "05_elipsa_uwiklana")


print("ZADANIE 6:")

a_val = 2
y_range = np.linspace(-5, 5, 100)
x_fill = np.linspace(-6, a_val, 100)
y_fill = np.linspace(-5, 5, 100)

fig, ax = setup_plot(f'Zadanie 6: Prosta x={a_val} i zbiór Z={{(b,y) : b ≤ {a_val}}}',
                      r'$x$', r'$y$')

# Zaznaczenie obszaru Z
Y_fill, X_fill = np.meshgrid(y_fill, x_fill)
ax.contourf(X_fill, Y_fill, np.ones_like(X_fill), levels=[0.5, 1.5], colors=['lightblue'], alpha=0.6)

# Prosta x=a
ax.axvline(x=a_val, color='red', linewidth=3, label=f'$x = {a_val}$')

ax.set_xlim(-6, 5)
ax.set_ylim(-5, 5)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "06_zbior_punktow")

# ============================================================================
# ZADANIE 7: Dwie krzywe parametryczne
# ============================================================================
print("ZADANIE 7: Krzywe parametryczne")

# Krzywa 1: Lissajous - x = sin(3t), y = sin(2t)
# Krzywa 2: Spirala - x = t*cos(t), y = t*sin(t), t ∈ [0, 4π]

t1 = np.linspace(0, 2*np.pi, 1000)
x_lissajous = np.sin(3*t1)
y_lissajous = np.sin(2*t1)

t2 = np.linspace(0, 4*np.pi, 1000)
x_spirala = t2 * np.cos(t2)
y_spirala = t2 * np.sin(t2)

# Osobne wykresy
fig, ax = setup_plot('Zadanie 7a: Krzywa Lissajousx (Krzywa 1)',
                      r'$x = \sin(3t)$', r'$y = \sin(2t)$')
ax.plot(x_lissajous, y_lissajous, 'b-', linewidth=2.5)
ax.set_aspect('equal')
save_figure(fig, "07a_lissajous")

fig, ax = setup_plot('Zadanie 7b: Spirala (Krzywa 2)',
                      r'$x = t\cos(t)$', r'$y = t\sin(t)$')
ax.plot(x_spirala, y_spirala, 'g-', linewidth=2.5)
save_figure(fig, "07b_spirala")

# Wspólny wykres
fig, ax = setup_plot('Zadanie 7c: Dwie krzywe parametryczne razem',
                      r'$x$', r'$y$')
ax.plot(x_lissajous, y_lissajous, 'b-', linewidth=2.5, label='Lissajous: sin(3t), sin(2t)')
ax.plot(x_spirala, y_spirala, 'g-', linewidth=2.5, label='Spirala: t·cos(t), t·sin(t)')
ax.legend(loc='best')
save_figure(fig, "07c_obie_krzywe")

# ============================================================================
# ZADANIE 8: Dwie funkcje dwóch zmiennych (3D)
# ============================================================================
print("ZADANIE 8: Funkcje dwóch zmiennych (3D)")

from mpl_toolkits.mplot3d import Axes3D

# Funkcja 1: z = x^2 + y^2 (paraboloida)
# Funkcja 2: z = sin(x)*cos(y) (función trygonometryczna)

x_3d = np.linspace(-3, 3, 100)
y_3d = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_3d, y_3d)

Z1 = X**2 + Y**2
Z2 = np.sin(X) * np.cos(Y)

# Osobny wykres dla funkcji 1
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_title('Zadanie 8a: Paraboloida $z = x^2 + y^2$', fontsize=13, fontweight='bold')
surf1 = ax.plot_surface(X, Y, Z1, cmap='viridis', alpha=0.9)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_zlabel('z', fontsize=11)
fig.colorbar(surf1, ax=ax, label='z')
plt.tight_layout()
save_figure(fig, "08a_paraboloida_3d")

# Osobny wykres dla funkcji 2
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_title(r'Zadanie 8b: Funkcja $z = \sin(x)\cos(y)$', fontsize=13, fontweight='bold')
surf2 = ax.plot_surface(X, Y, Z2, cmap='plasma', alpha=0.9)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_zlabel('z', fontsize=11)
fig.colorbar(surf2, ax=ax, label='z')
plt.tight_layout()
save_figure(fig, "08b_sin_cos_3d")

# Wspólny wykres (subploty)
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title('Paraboloida: $z = x^2 + y^2$', fontsize=12, fontweight='bold')
ax1.plot_surface(X, Y, Z1, cmap='viridis', alpha=0.9)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

ax2 = fig.add_subplot(122, projection='3d')
ax2.set_title(r'Funkcja: $z = \sin(x)\cos(y)$', fontsize=12, fontweight='bold')
ax2.plot_surface(X, Y, Z2, cmap='plasma', alpha=0.9)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')

plt.tight_layout()
save_figure(fig, "08c_obie_3d")

print("\n✓ Zadania 1-8 ukończone")


ZADANIE 1: Funkcja z ekstremami lokalnymi
✓ Zapisano: 01_ekstrema
ZADANIE 2: Trzy funkcje z ograniczonymi zakresami
✓ Zapisano: 02_trzy_funkcje
ZADANIE 3: Funkcja klamerkowa i jej pochodna
✓ Zapisano: 03_klamerkowa
ZADANIE 4: Trójkąt utworzony przez trzy proste
✓ Zapisano: 04_trojkat
ZADANIE 5: Krzywa uwikłana (elipsa)
✓ Zapisano: 05_elipsa_uwiklana
ZADANIE 6: Prosta x=a i zbiór punktów Z={(b,y): b ≤ a}
✓ Zapisano: 06_zbior_punktow
ZADANIE 7: Krzywe parametryczne
✓ Zapisano: 07a_lissajous
✓ Zapisano: 07b_spirala
✓ Zapisano: 07c_obie_krzywe
ZADANIE 8: Funkcje dwóch zmiennych (3D)
✓ Zapisano: 08a_paraboloida_3d
✓ Zapisano: 08b_sin_cos_3d
✓ Zapisano: 08c_obie_3d

✓ Zadania 1-8 ukończone


In [7]:
# ============================================================================
# ZADANIA 9-15 (POWTÓRZENIE Z NIESTANDARDOWYMI PARAMETRAMI)
# ============================================================================
# Każde zadanie powtórzone z zmienionymi parametrami (zakres, kolory, osie, itp.)
# ============================================================================

print("\n" + "="*70)
print("ZADANIA 9-15: POWTÓRZENIE Z NIESTANDARDOWYMI PARAMETRAMI")
print("="*70)

# ============================================================================
# ZADANIE 9 (powtórzenie zad. 2): Trzy funkcje - ZMIENIONE: zakresy, logskala
# ============================================================================
print("\nZADANIE 9 (wariant zad. 2): Inne funkcje, inny zakres")

# ZMIANA: inne funkcje, inny zakres, bez siatki na jednej osi
functions_9 = {
    'f1': {'expr': sp.sin(x), 'range': [0, 2*np.pi], 'color': 'purple', 'label': r'$\sin(x), x \in [0, 2\pi]$'},
    'f2': {'expr': sp.exp(x/2), 'range': [-2, 2], 'color': 'orange', 'label': r'$e^{x/2}, x \in [-2, 2]$'},
    'f3': {'expr': sp.sqrt(x), 'range': [0.1, 10], 'color': 'brown', 'label': r'$\sqrt{x}, x \in [0.1, 10]$'}
}

fig, ax = setup_plot('Zadanie 9: Wariant zad. 2 - inne funkcje i zakresy',
                      r'$x$', r'$y$')

for key, func_dict in functions_9.items():
    x_range = func_dict['range']
    x_vals = np.linspace(x_range[0], x_range[1], 200)
    y_vals = sympy_to_numpy(func_dict['expr'], x, x_vals)
    ax.plot(x_vals, y_vals, color=func_dict['color'], linewidth=2.5, label=func_dict['label'], linestyle='--')

ax.axhline(y=0, color='k', linewidth=0.5)
ax.grid(False)  # Brak siatki - różnica parametru
ax.legend(loc='best', fontsize=9)
save_figure(fig, "09_wariant_2")

# ============================================================================
# ZADANIE 10 (powtórzenie zad. 3): Funkcja klamerkowa - ZMIANA: inny kształt
# ============================================================================
print("ZADANIE 10 (wariant zad. 3): Inna funkcja klamerkowa")

# Nowa funkcja: exp(x) dla x < 0, 1 dla x = 0, ln(x+1) dla x > 0
x_left_10 = np.linspace(-3, 0, 200)
x_right_10 = np.linspace(0, 5, 200)

y_left_10 = np.exp(x_left_10)
y_right_10 = np.log(x_right_10 + 1)

y_prime_left_10 = np.exp(x_left_10)
y_prime_right_10 = 1 / (x_right_10 + 1)

fig, ax = setup_plot('Zadanie 10: Wariant zad. 3 - inna funkcja klamerkowa',
                      r'$x$', r'$y$')

ax.plot(x_left_10, y_left_10, 'b-', linewidth=2.5, label=r"$e^x$ dla $x < 0$")
ax.plot(x_right_10, y_right_10, 'b-', linewidth=2.5, label=r"$\ln(x+1)$ dla $x \geq 0$")
ax.scatter([0], [1], color='blue', s=100, zorder=5)

ax.plot(x_left_10, y_prime_left_10, 'r--', linewidth=2.5, alpha=0.7, label=r"$(e^x)'$")
ax.plot(x_right_10, y_prime_right_10, 'r--', linewidth=2.5, alpha=0.7, label=r"$(\ln(x+1))'$")

ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlim(-3.5, 5.5)
ax.legend(loc='best', fontsize=9)
save_figure(fig, "10_wariant_3")

# ============================================================================
# ZADANIE 11 (powtórzenie zad. 4): Trójkąt - ZMIANA: inne proste, wypełnienie
# ============================================================================
print("ZADANIE 11 (wariant zad. 4): Inne proste, inny trójkąt")

# Trzy nowe proste: y = -0.5x + 2, y = 2x - 1, y = -x
# Przecięcia:
# l1 i l2: -0.5x + 2 = 2x - 1 → 3 = 2.5x → x = 1.2, y = 1.4
# l2 i l3: 2x - 1 = -x → 3x = 1 → x = 1/3, y = -1/3
# l1 i l3: -0.5x + 2 = -x → 0.5x = -2 → x = -4, y = 4

vertices_11 = np.array([[1.2, 1.4], [1/3, -1/3], [-4, 4]])

fig, ax = setup_plot('Zadanie 11: Wariant zad. 4 - inne proste',
                      r'$x$', r'$y$')

x_range_11 = np.linspace(-5, 2, 200)

y_l1_11 = -0.5 * x_range_11 + 2
y_l2_11 = 2 * x_range_11 - 1
y_l3_11 = -x_range_11

ax.plot(x_range_11, y_l1_11, 'b-', linewidth=2.5, label=r'$y = -0.5x + 2$')
ax.plot(x_range_11, y_l2_11, 'g-', linewidth=2.5, label=r'$y = 2x - 1$')
ax.plot(x_range_11, y_l3_11, 'r-', linewidth=2.5, label=r'$y = -x$')

triangle_11 = Polygon(vertices_11, facecolor='cyan', alpha=0.5, edgecolor='black', linewidth=2)
ax.add_patch(triangle_11)

ax.scatter(vertices_11[:, 0], vertices_11[:, 1], color='red', s=100, zorder=5, marker='s')

ax.set_xlim(-5, 2)
ax.set_ylim(-2, 5)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "11_wariant_4")

# ============================================================================
# ZADANIE 12 (powtórzenie zad. 5): Krzywa uwikłana - ZMIANA: hiperbola
# ============================================================================
print("ZADANIE 12 (wariant zad. 5): Hiperbal zamiast elipsy")

# Hiperbola: x^2/4 - y^2/9 = 1
# Parametryzacja: x = 2*sec(t), y = 3*tan(t) lub bardziej praktycznie:
# x = 2*cosh(t), y = 3*sinh(t)

t_hyperbola = np.linspace(-3, 3, 500)
x_hyperbola_right = 2 * np.cosh(t_hyperbola)
y_hyperbola_right = 3 * np.sinh(t_hyperbola)
x_hyperbola_left = -2 * np.cosh(t_hyperbola)
y_hyperbola_left = 3 * np.sinh(t_hyperbola)

fig, ax = setup_plot(r'Zadanie 12: Wariant zad. 5 - Hiperbola $\frac{x^2}{4} - \frac{y^2}{9} = 1$',
                      r'$x$', r'$y$')

ax.plot(x_hyperbola_right, y_hyperbola_right, 'purple', linewidth=2.5, label='Hiperbola')
ax.plot(x_hyperbola_left, y_hyperbola_left, 'purple', linewidth=2.5)
ax.scatter([0], [0], color='red', s=100, marker='+', linewidths=2, label='Środek')

ax.set_xlim(-8, 8)
ax.set_ylim(-10, 10)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.grid(True, alpha=0.3, linestyle=':')  # Linia przerywana - różnica parametru
ax.legend(loc='best')
save_figure(fig, "12_wariant_5_hiperbola")

# ============================================================================
# ZADANIE 13 (powtórzenie zad. 6): Zbiór punktów - ZMIANA: zakresy, orientacja
# ============================================================================
print("ZADANIE 13 (wariant zad. 6): Zbiór punktów - inna konfiguracja")

# Zamiast x ≤ a, robimy y ≤ b
a_val_13 = 1.5
b_val_13 = 3

y_fill_13 = np.linspace(-5, b_val_13, 100)
x_fill_13 = np.linspace(-6, 6, 100)

fig, ax = setup_plot(f'Zadanie 13: Wariant zad. 6 - Zbiór {{(x,y) : y ≤ {b_val_13}}}',
                      r'$x$', r'$y$')

Y_fill_13, X_fill_13 = np.meshgrid(y_fill_13, x_fill_13)
ax.contourf(X_fill_13, Y_fill_13, np.ones_like(X_fill_13), levels=[0.5, 1.5], 
            colors=['lightcoral'], alpha=0.6, label=f'$y \\leq {b_val_13}$')

ax.axhline(y=b_val_13, color='darkred', linewidth=3, label=f'$y = {b_val_13}$')

ax.set_xlim(-6, 6)
ax.set_ylim(-5, 5)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "13_wariant_6")

# ============================================================================
# ZADANIE 14 (powtórzenie zad. 7): Krzywe parametryczne - ZMIANA: inne krzywe
# ============================================================================
print("ZADANIE 14 (wariant zad. 7): Inne krzywe parametryczne")

# Krzywa 1: Róża - x = cos(5t)*cos(t), y = cos(5t)*sin(t)
# Krzywa 2: Epicykloida - x = (R+r)*cos(t) - r*cos((R+r)/r*t), y = (R+r)*sin(t) - r*sin((R+r)/r*t)
# Uproszczona epicykloida z R=5, r=1

t_rose = np.linspace(0, 2*np.pi, 1000)
x_rose = np.cos(5*t_rose) * np.cos(t_rose)
y_rose = np.cos(5*t_rose) * np.sin(t_rose)

t_epi = np.linspace(0, 8*np.pi, 1000)
R, r = 5, 1
x_epi = (R+r)*np.cos(t_epi) - r*np.cos((R+r)/r*t_epi)
y_epi = (R+r)*np.sin(t_epi) - r*np.sin((R+r)/r*t_epi)

# Osobne wykresy
fig, ax = setup_plot('Zadanie 14a: Róża (krzywa walentynki)',
                      r'$x = \cos(5t)\cos(t)$', r'$y = \cos(5t)\sin(t)$')
ax.plot(x_rose, y_rose, 'hotpink', linewidth=2.5)
ax.set_aspect('equal')
save_figure(fig, "14a_rosa")

fig, ax = setup_plot('Zadanie 14b: Epicykloida',
                      r'$x$', r'$y$')
ax.plot(x_epi, y_epi, 'darkblue', linewidth=2.5)
ax.set_aspect('equal')
save_figure(fig, "14b_epicykloida")

# Wspólny wykres
fig, ax = setup_plot('Zadanie 14c: Dwie krzywe parametryczne (wariant)',
                      r'$x$', r'$y$')
ax.plot(x_rose, y_rose, 'hotpink', linewidth=2.5, label='Róża')
ax.plot(x_epi, y_epi, 'darkblue', linewidth=2.5, label='Epicykloida')
ax.legend(loc='best')
save_figure(fig, "14c_obie_wariant")

# ============================================================================
# ZADANIE 15 (powtórzenie zad. 8): 3D - ZMIANA: inne funkcje, kolory
# ============================================================================
print("ZADANIE 15 (wariant zad. 8): Inne funkcje 3D")

# Funkcja 1: z = sin(sqrt(x^2 + y^2)) (fale koncentryczne)
# Funkcja 2: z = x*y (siodło)

x_3d_15 = np.linspace(-5, 5, 100)
y_3d_15 = np.linspace(-5, 5, 100)
X_15, Y_15 = np.meshgrid(x_3d_15, y_3d_15)

Z1_15 = np.sin(np.sqrt(X_15**2 + Y_15**2))
Z2_15 = X_15 * Y_15

# Osobne wykresy
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_title(r'Zadanie 15a: Fale $z = \sin(\sqrt{x^2 + y^2})$', fontsize=13, fontweight='bold')
surf1_15 = ax.plot_surface(X_15, Y_15, Z1_15, cmap='coolwarm', alpha=0.9)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
fig.colorbar(surf1_15, ax=ax, label='z')
plt.tight_layout()
save_figure(fig, "15a_fale_3d")

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_title(r'Zadanie 15b: Siodło $z = xy$', fontsize=13, fontweight='bold')
surf2_15 = ax.plot_surface(X_15, Y_15, Z2_15, cmap='RdBu', alpha=0.9)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
fig.colorbar(surf2_15, ax=ax, label='z')
plt.tight_layout()
save_figure(fig, "15b_siedlo_3d")

# Wspólny wykres
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title(r'Fale: $z = \sin(\sqrt{x^2 + y^2})$', fontsize=12, fontweight='bold')
ax1.plot_surface(X_15, Y_15, Z1_15, cmap='coolwarm', alpha=0.9)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

ax2 = fig.add_subplot(122, projection='3d')
ax2.set_title(r'Siodło: $z = xy$', fontsize=12, fontweight='bold')
ax2.plot_surface(X_15, Y_15, Z2_15, cmap='RdBu', alpha=0.9)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')

plt.tight_layout()
save_figure(fig, "15c_obie_3d_wariant")

print("\n✓ Zadania 9-15 ukończone")


ZADANIA 9-15: POWTÓRZENIE Z NIESTANDARDOWYMI PARAMETRAMI

ZADANIE 9 (wariant zad. 2): Inne funkcje, inny zakres
✓ Zapisano: 09_wariant_2
ZADANIE 10 (wariant zad. 3): Inna funkcja klamerkowa
✓ Zapisano: 10_wariant_3
ZADANIE 11 (wariant zad. 4): Inne proste, inny trójkąt
✓ Zapisano: 11_wariant_4
ZADANIE 12 (wariant zad. 5): Hiperbal zamiast elipsy
✓ Zapisano: 12_wariant_5_hiperbola
ZADANIE 13 (wariant zad. 6): Zbiór punktów - inna konfiguracja


C:\Users\Tomek\AppData\Local\Temp\ipykernel_52420\1766661174.py:152: UserWarning: The following kwargs were not used by contour: 'label'
  ax.contourf(X_fill_13, Y_fill_13, np.ones_like(X_fill_13), levels=[0.5, 1.5],


✓ Zapisano: 13_wariant_6
ZADANIE 14 (wariant zad. 7): Inne krzywe parametryczne
✓ Zapisano: 14a_rosa
✓ Zapisano: 14b_epicykloida
✓ Zapisano: 14c_obie_wariant
ZADANIE 15 (wariant zad. 8): Inne funkcje 3D
✓ Zapisano: 15a_fale_3d
✓ Zapisano: 15b_siedlo_3d
✓ Zapisano: 15c_obie_3d_wariant

✓ Zadania 9-15 ukończone


In [8]:
# ============================================================================
# ZADANIA 16-21 (ZAAWANSOWANE)
# ============================================================================

print("\n" + "="*70)
print("ZADANIA 16-21: ZAAWANSOWANE")
print("="*70)

# ============================================================================
# ZADANIE 16: Pięć punktów w R^2 + krzywa przez nie
# ============================================================================
print("\nZADANIE 16: Pięć punktów i krzywa interpolacyjna")

# Pięć punktów
points_16 = np.array([[1, 2], [2, 3], [3, 5], [4, 4], [5, 1]])

# Fit splajn przez punkty
tck, u = splprep(points_16.T, s=1, k=3)
u_new = np.linspace(0, 1, 300)
x_spline, y_spline = splev(u_new, tck)

# Osobny plot dla punktów
fig, ax = setup_plot('Zadanie 16a: Pięć punktów w R²',
                      r'$x$', r'$y$')
ax.scatter(points_16[:, 0], points_16[:, 1], color='red', s=150, marker='o', zorder=5, label='Punkty')
for i, pt in enumerate(points_16):
    ax.annotate(f'P{i+1}({pt[0]},{pt[1]})', xy=pt, xytext=(5, 5), 
                textcoords='offset points', fontsize=9)
ax.set_xlim(0, 6)
ax.set_ylim(0, 6)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "16a_punkty")

# Plot z krzywą interpolacyjną
fig, ax = setup_plot('Zadanie 16b: Punkty połączone krzywą interpolacyjną',
                      r'$x$', r'$y$')
ax.plot(x_spline, y_spline, 'b-', linewidth=2.5, label='Krzywa interpolacyjna')
ax.scatter(points_16[:, 0], points_16[:, 1], color='red', s=150, marker='o', 
           zorder=5, label='Punkty')
ax.set_xlim(0, 6)
ax.set_ylim(0, 6)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.legend(loc='best')
save_figure(fig, "16b_krzywa_przez_punkty")

# ============================================================================
# ZADANIE 17: Krzywa 3D w przestrzeni
# ============================================================================
print("ZADANIE 17: Krzywa w przestrzeni 3D")

# Helisa: x = cos(t), y = sin(t), z = t
t_3d = np.linspace(0, 6*np.pi, 500)
x_helix = np.cos(t_3d)
y_helix = np.sin(t_3d)
z_helix = t_3d

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_title('Zadanie 17: Krzywa 3D - Helisa\n' + 
             r'$x = \cos(t), y = \sin(t), z = t$', fontsize=13, fontweight='bold')
ax.plot(x_helix, y_helix, z_helix, 'b-', linewidth=2.5)
ax.scatter([1], [0], [0], color='red', s=100, marker='o', label='Start')
ax.scatter([np.cos(6*np.pi)], [np.sin(6*np.pi)], [6*np.pi], color='green', 
           s=100, marker='s', label='Koniec')
ax.set_xlabel(r'$x = \cos(t)$', fontsize=11)
ax.set_ylabel(r'$y = \sin(t)$', fontsize=11)
ax.set_zlabel(r'$z = t$', fontsize=11)
ax.legend(loc='best')
plt.tight_layout()
save_figure(fig, "17_helisa_3d")

# ============================================================================
# ZADANIE 18: Elipsoida lub hiperboloida z opisem
# ============================================================================
print("ZADANIE 18: Elipsoida")

# Elipsoida: x^2/a^2 + y^2/b^2 + z^2/c^2 = 1
# Parametryzacja: x = a*sin(u)*cos(v), y = b*sin(u)*sin(v), z = c*cos(u)

a, b, c = 3, 2, 1.5
u_ellipsoid = np.linspace(0, np.pi, 50)
v_ellipsoid = np.linspace(0, 2*np.pi, 50)
u_mesh, v_mesh = np.meshgrid(u_ellipsoid, v_ellipsoid)

x_ellipsoid = a * np.sin(u_mesh) * np.cos(v_mesh)
y_ellipsoid = b * np.sin(u_mesh) * np.sin(v_mesh)
z_ellipsoid = c * np.cos(u_mesh)

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection='3d')

ax.set_title('Zadanie 18: Elipsoida\n' + 
             fr'$\frac{{x^2}}{{{a}^2}} + \frac{{y^2}}{{{b}^2}} + \frac{{z^2}}{{{c}^2}} = 1$',
             fontsize=13, fontweight='bold')

surf = ax.plot_surface(x_ellipsoid, y_ellipsoid, z_ellipsoid, cmap='viridis', alpha=0.8)

ax.set_xlabel(f'x (a={a})', fontsize=11)
ax.set_ylabel(f'y (b={b})', fontsize=11)
ax.set_zlabel(f'z (c={c})', fontsize=11)

fig.colorbar(surf, ax=ax, label='Powierzchnia')

# Tekst z opisem
description = f"Elipsoida o równaniu:\n" \
              f"x²/{a}² + y²/{b}² + z²/{c}² = 1\n\n" \
              f"Parametry:\n" \
              f"  a = {a} (półoś x)\n" \
              f"  b = {b} (półoś y)\n" \
              f"  c = {c} (półoś z)\n\n" \
              f"Powierzchnia elipsoidy jest generowana\n" \
              f"przy użyciu parametryzacji sferycznej."

ax.text2D(0.02, 0.98, description, transform=ax.transAxes, fontsize=10,
          verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
          family='monospace')

plt.tight_layout()
save_figure(fig, "18_elipsoida")

# ============================================================================
# ZADANIE 19: Funkcja 2-zmiennych + warstwice
# ============================================================================
print("ZADANIE 19: Funkcja 2-zmiennych i jej warstwice")

# Wybrana funkcja: z = exp(-(x^2 + y^2))

x_func19 = np.linspace(-3, 3, 100)
y_func19 = np.linspace(-3, 3, 100)
X_19, Y_19 = np.meshgrid(x_func19, y_func19)
Z_19 = np.exp(-(X_19**2 + Y_19**2))

# Osobne wykresy
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_title(r'Zadanie 19a: Funkcja 2-zmiennych $z = e^{-(x^2+y^2)}$',
             fontsize=13, fontweight='bold')
surf19 = ax.plot_surface(X_19, Y_19, Z_19, cmap='hot', alpha=0.9)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_zlabel('z', fontsize=11)
fig.colorbar(surf19, ax=ax, label='z')
plt.tight_layout()
save_figure(fig, "19a_funkcja_3d")

# Warstwice
fig, ax = setup_plot(r'Zadanie 19b: Warstwice funkcji $z = e^{-(x^2+y^2)}$',
                      r'$x$', r'$y$')
contour = ax.contourf(X_19, Y_19, Z_19, levels=20, cmap='hot')
contour_lines = ax.contour(X_19, Y_19, Z_19, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax.clabel(contour_lines, inline=True, fontsize=8, fmt='%1.2f')
cbar = plt.colorbar(contour, ax=ax, label='z')
ax.set_aspect('equal')
save_figure(fig, "19b_warstwice")

# Wspólny plot (3D + warstwice na subplotach)
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title(r'3D: $z = e^{-(x^2+y^2)}$', fontsize=12, fontweight='bold')
ax1.plot_surface(X_19, Y_19, Z_19, cmap='hot', alpha=0.9)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

ax2 = fig.add_subplot(122)
ax2.set_title(r'Warstwice', fontsize=12, fontweight='bold')
contourf2 = ax2.contourf(X_19, Y_19, Z_19, levels=20, cmap='hot')
contour_lines2 = ax2.contour(X_19, Y_19, Z_19, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax2.clabel(contour_lines2, inline=True, fontsize=8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.colorbar(contourf2, ax=ax2, label='z')
ax2.set_aspect('equal')

plt.tight_layout()
save_figure(fig, "19c_funkcja_i_warstwice")

# ============================================================================
# ZADANIE 20: Subploty
# ============================================================================
print("ZADANIE 20: Superpozycje wykreśów (subploty)")

# Part A: Trzy wykresy obok siebie
fig = plt.figure(figsize=(18, 5))

# Subplot 1: Paraboloida
ax1 = fig.add_subplot(131, projection='3d')
ax1.set_title('Paraboloida', fontsize=12, fontweight='bold')
ax1.plot_surface(X, Y, Z1, cmap='viridis', alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

# Subplot 2: sin(x)*cos(y)
ax2 = fig.add_subplot(132, projection='3d')
ax2.set_title(r'$\sin(x)\cos(y)$', fontsize=12, fontweight='bold')
ax2.plot_surface(X, Y, Z2, cmap='plasma', alpha=0.8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')

# Subplot 3: Fale konc.
ax3 = fig.add_subplot(133, projection='3d')
ax3.set_title(r'Fale $\sin(\sqrt{x^2+y^2})$', fontsize=12, fontweight='bold')
ax3.plot_surface(X_15, Y_15, Z1_15, cmap='coolwarm', alpha=0.8)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_zlabel('z')

plt.tight_layout()
save_figure(fig, "20a_trzy_wykresy")

# Part B: Cztery wykresy (2x2)
fig = plt.figure(figsize=(14, 12))

# Subplot 1: Paraboloida
ax1 = fig.add_subplot(221, projection='3d')
ax1.set_title('Paraboloida: $z = x^2 + y^2$', fontsize=11, fontweight='bold')
ax1.plot_surface(X, Y, Z1, cmap='viridis', alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')

# Subplot 2: sin(x)*cos(y)
ax2 = fig.add_subplot(222, projection='3d')
ax2.set_title(r'$\sin(x)\cos(y)$', fontsize=11, fontweight='bold')
ax2.plot_surface(X, Y, Z2, cmap='plasma', alpha=0.8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')

# Subplot 3: Fale
ax3 = fig.add_subplot(223, projection='3d')
ax3.set_title(r'Fale: $\sin(\sqrt{x^2+y^2})$', fontsize=11, fontweight='bold')
ax3.plot_surface(X_15, Y_15, Z1_15, cmap='coolwarm', alpha=0.8)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_zlabel('z')

# Subplot 4: Siodło
ax4 = fig.add_subplot(224, projection='3d')
ax4.set_title(r'Siodło: $z = xy$', fontsize=11, fontweight='bold')
ax4.plot_surface(X_15, Y_15, Z2_15, cmap='RdBu', alpha=0.8)
ax4.set_xlabel('x')
ax4.set_ylabel('y')
ax4.set_zlabel('z')

plt.tight_layout()
save_figure(fig, "20b_cztery_wykresy_2x2")

# ============================================================================
# ZADANIE 21: Eksport wybranego wykresu w różnych formatach
# ============================================================================
print("ZADANIE 21: Eksport wykresu w formatach PDF, JPG, PNG")

# Wybieramy wcześniejszy plots - elipsoidę z zadania 18
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

ax.set_title('Zadanie 21: Eksport przykładowego wykresu (Elipsoida)\n' + 
             fr'$\frac{{x^2}}{{{a}^2}} + \frac{{y^2}}{{{b}^2}} + \frac{{z^2}}{{{c}^2}} = 1$',
             fontsize=13, fontweight='bold')

surf = ax.plot_surface(x_ellipsoid, y_ellipsoid, z_ellipsoid, cmap='viridis', alpha=0.85)

ax.set_xlabel(f'x (a={a})', fontsize=11)
ax.set_ylabel(f'y (b={b})', fontsize=11)
ax.set_zlabel(f'z (c={c})', fontsize=11)

fig.colorbar(surf, ax=ax, label='Powierzchnia')

# Zapis w trzech formatach
pdf_path_21 = OUTPUT_DIR / "21_export_elipsoida.pdf"
png_path_21 = OUTPUT_DIR / "21_export_elipsoida.png"
jpg_path_21 = OUTPUT_DIR / "21_export_elipsoida.jpg"

fig.savefig(pdf_path_21, bbox_inches='tight', dpi=300)
fig.savefig(png_path_21, bbox_inches='tight', dpi=200)
fig.savefig(jpg_path_21, bbox_inches='tight', dpi=200)

plt.close(fig)

print(f"✓ Zadanie 21 - zapapisano:")
print(f"  - {pdf_path_21.name} (PDF, dpi=300)")
print(f"  - {png_path_21.name} (PNG, dpi=200)")
print(f"  - {jpg_path_21.name} (JPG, dpi=200)")

print("\n✓ Zadania 16-21 ukończone")


ZADANIA 16-21: ZAAWANSOWANE

ZADANIE 16: Pięć punktów i krzywa interpolacyjna
✓ Zapisano: 16a_punkty
✓ Zapisano: 16b_krzywa_przez_punkty
ZADANIE 17: Krzywa w przestrzeni 3D
✓ Zapisano: 17_helisa_3d
ZADANIE 18: Elipsoida
✓ Zapisano: 18_elipsoida
ZADANIE 19: Funkcja 2-zmiennych i jej warstwice
✓ Zapisano: 19a_funkcja_3d
✓ Zapisano: 19b_warstwice
✓ Zapisano: 19c_funkcja_i_warstwice
ZADANIE 20: Superpozycje wykreśów (subploty)
✓ Zapisano: 20a_trzy_wykresy
✓ Zapisano: 20b_cztery_wykresy_2x2
ZADANIE 21: Eksport wykresu w formatach PDF, JPG, PNG
✓ Zadanie 21 - zapapisano:
  - 21_export_elipsoida.pdf (PDF, dpi=300)
  - 21_export_elipsoida.png (PNG, dpi=200)
  - 21_export_elipsoida.jpg (JPG, dpi=200)

✓ Zadania 16-21 ukończone


In [9]:
# ============================================================================
# PODSUMOWANIE I KOMPILACJA PDF
# ============================================================================

print("\n" + "="*70)
print("PODSUMOWANIE")
print("="*70)

import os
from pathlib import Path

# Policzymy ile plików zostało wygenerowanych
output_files = list(OUTPUT_DIR.glob("*.pdf")) + list(OUTPUT_DIR.glob("*.png")) + list(OUTPUT_DIR.glob("*.jpg"))
output_files.sort()

print(f"\n✓ Wszystkie wykresy zostały wygenerowane!")
print(f"\n📊 Liczba plików: {len(output_files)}")
print(f"📁 Lokalizacja: {OUTPUT_DIR.absolute()}\n")

print("Wygenerowane pliki:")
for i, file in enumerate(output_files, 1):
    print(f"  {i:2d}. {file.name}")

print("\n" + "="*70)
print("SZCZEGÓŁY ZADAŃ")
print("="*70)

summary = """
ZADANIA 1-8 (Wersja podstawowa):
  1. Funkcja z ekstremami (x⁴-4x²) - zaznaczone ekstrema
  2. Trzy funkcje na jednym wykresie - różne kolory i zakresy
  3. Funkcja klamerkowa + jej pochodna
  4. Trójkąt z trzech prostych - wypełniony interior
  5. Krzywa uwikłana - elipsa
  6. Prosta x=a i zbiór punktów Z={(b,y): b≤a}
  7. Dwie krzywe parametryczne (Lissajous + Spirala)
  8. Dwie funkcje 2-zmiennych (3D) - Paraboloida + sin(x)cos(y)

ZADANIA 9-15 (Wersja ze zmienionymi parametrami):
  9. Wariant zad. 2 - inne funkcje (sin, exp, sqrt), bez siatki
  10. Wariant zad. 3 - inna funkcja klamerkowa (exp, ln)
  11. Wariant zad. 4 - inne proste, inny trójkąt
  12. Wariant zad. 5 - hiperbola zamiast elipsy
  13. Wariant zad. 6 - zbiór o orientacji y
  14. Wariant zad. 7 - inne krzywe (Róża + Epicykloida)
  15. Wariant zad. 8 - inne 3D (Fale + Siodło)

ZADANIA 16-21 (Zaawansowane):
  16. Pięć punktów + krzywa interpolacyjna
  17. Krzywa 3D - Helisa
  18. Elipsoida z parametrami i opisem
  19. Funkcja 2-zmiennych (exp) + warstwice
  20. Subploty: trzy wykresy + cztery wykresy (2x2)
  21. Eksport wybranego wykresu (Elipsoida) w PDF, PNG, JPG

KAŻDY WYKRES ZAWIERA:
  ✓ Tytuł (boldface)
  ✓ Podpisane osie (x, y, z gdy dotyczy)
  ✓ Legendę (gdzie dotyczy)
  ✓ Sietkę (gdzie dotyczy)
  ✓ Wzory matematyczne (LaTeX w tytułach i legendach)
"""

print(summary)

print("="*70)
print("ARCHITEKTURA KODU")
print("="*70)

arch = """
Kod wykorzystuje:
  1. Słowniki do przechowywania funkcji (klucz=ID, wartość=funkcja)
  2. SymPy do obliczeń symbolicznych (pochodne, ekstrema)
  3. NumPy do obliczeń numerycznych (wektory, macierze)
  4. Matplotlib do rysowania (2D i 3D)
  5. SciPy do interpolacji (splprep, splev dla splajnów)

Struktura:
  - Komórka 1: Importy + helper functions
  - Komórka 2: Zadania 1-8
  - Komórka 3: Zadania 9-15
  - Komórka 4: Zadania 16-21
  - Komórka 5: Podsumowanie (ten plik)

Helper functions:
  - setup_plot(title, xlabel, ylabel) - tworzy figurę z formatowaniem
  - save_figure(fig, filename_base) - zapis PDF/PNG/JPG
  - sympy_to_numpy(expr, var, x_vals) - konwersja wyrażeń
  - find_critical_points(f, x_var, a, b) - znajdowanie ekstremów
"""

print(arch)

print("="*70)
print("✓ SKONCZONO! Wszystkie 21 zadań zostało wykonane.")
print("="*70)


PODSUMOWANIE

✓ Wszystkie wykresy zostały wygenerowane!

📊 Liczba plików: 3
📁 Lokalizacja: c:\Users\Tomek\Downloads\Obliczenia_Naukowe_2026\Obliczenia_Naukowe_2026\Lista 3\wykresy_wyjscie

Wygenerowane pliki:
   1. 21_export_elipsoida.jpg
   2. 21_export_elipsoida.pdf
   3. 21_export_elipsoida.png

SZCZEGÓŁY ZADAŃ

ZADANIA 1-8 (Wersja podstawowa):
  1. Funkcja z ekstremami (x⁴-4x²) - zaznaczone ekstrema
  2. Trzy funkcje na jednym wykresie - różne kolory i zakresy
  3. Funkcja klamerkowa + jej pochodna
  4. Trójkąt z trzech prostych - wypełniony interior
  5. Krzywa uwikłana - elipsa
  6. Prosta x=a i zbiór punktów Z={(b,y): b≤a}
  7. Dwie krzywe parametryczne (Lissajous + Spirala)
  8. Dwie funkcje 2-zmiennych (3D) - Paraboloida + sin(x)cos(y)

ZADANIA 9-15 (Wersja ze zmienionymi parametrami):
  9. Wariant zad. 2 - inne funkcje (sin, exp, sqrt), bez siatki
  10. Wariant zad. 3 - inna funkcja klamerkowa (exp, ln)
  11. Wariant zad. 4 - inne proste, inny trójkąt
  12. Wariant zad. 5 - 

In [10]:
# ============================================================================
# MERGE WSZYSTKICH PDF W JEDEN PLIK - Automatycznie
# ============================================================================

from PyPDF2 import PdfMerger
from pathlib import Path

print("\n" + "="*70)
print("TWORZENIE SKOMPILOWANEGO PDF")
print("="*70)

# OUTPUT_DIR i SUBFOLDERS są już zdefiniowane w komórce 1

# Odczyt PDF-ów z PDF_indywidualne/, posortowanych po numerze
pdf_files = sorted([f for f in SUBFOLDERS['pdf'].glob("*.pdf")])

if pdf_files:
    # PdfMerger
    merger = PdfMerger()
    
    print(f"\nŁączenie {len(pdf_files)} PDF-ów:\n")
    
    for pdf_file in pdf_files:
        merger.append(str(pdf_file))
        print(f"  + {pdf_file.name}")
    
    # Zapis połączonego PDF - bezpośrednio do PDF_do_odesłania/
    merged_pdf_path = SUBFOLDERS['main_pdf'] / "LISTA3_WSZYSTKIE_WYKRESY.pdf"
    merger.write(str(merged_pdf_path))
    merger.close()
    
    print(f"\n✓ Skompilowany PDF: {merged_pdf_path.name}")
    print(f"  Lokalizacja: {merged_pdf_path.absolute()}")
    print(f"  Rozmiar: {merged_pdf_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("⚠️  Brak PDF-ów do scalenia!")


TWORZENIE SKOMPILOWANEGO PDF

Łączenie 32 PDF-ów:

  + 01_ekstrema.pdf
  + 02_trzy_funkcje.pdf
  + 03_klamerkowa.pdf
  + 04_trojkat.pdf
  + 05_elipsa_uwiklana.pdf
  + 06_zbior_punktow.pdf
  + 07a_lissajous.pdf
  + 07b_spirala.pdf
  + 07c_obie_krzywe.pdf
  + 08a_paraboloida_3d.pdf
  + 08b_sin_cos_3d.pdf
  + 08c_obie_3d.pdf
  + 09_wariant_2.pdf
  + 10_wariant_3.pdf
  + 11_wariant_4.pdf
  + 12_wariant_5_hiperbola.pdf
  + 13_wariant_6.pdf
  + 14a_rosa.pdf
  + 14b_epicykloida.pdf
  + 14c_obie_wariant.pdf
  + 15a_fale_3d.pdf
  + 15b_siedlo_3d.pdf
  + 15c_obie_3d_wariant.pdf
  + 16a_punkty.pdf
  + 16b_krzywa_przez_punkty.pdf
  + 17_helisa_3d.pdf
  + 18_elipsoida.pdf
  + 19a_funkcja_3d.pdf
  + 19b_warstwice.pdf
  + 19c_funkcja_i_warstwice.pdf
  + 20a_trzy_wykresy.pdf
  + 20b_cztery_wykresy_2x2.pdf

✓ Skompilowany PDF: LISTA3_WSZYSTKIE_WYKRESY.pdf
  Lokalizacja: c:\Users\Tomek\Downloads\Obliczenia_Naukowe_2026\Obliczenia_Naukowe_2026\Lista 3\wykresy_wyjscie\PDF_do_odesłania\LISTA3_WSZYSTKIE_WY

In [11]:
# ============================================================================
# PODSUMOWANIE - Struktura plików i statystyka
# ============================================================================

print("\n" + "="*70)
print("📊 PODSUMOWANIE - STRUKTURA PLIKÓW")
print("="*70)

print("\n✅ WSZYSTKIE 21 ZADAŃ UKOŃCZONE!\n")

# Wyświetl strukturę i statystyki
print("Struktura folderów:")
print()

folder_stats = {}
for folder_name, folder_path in SUBFOLDERS.items():
    if folder_path.exists():
        files_in_folder = list(folder_path.glob("*"))
        file_count = len([f for f in files_in_folder if f.is_file()])
        total_size = sum(f.stat().st_size for f in files_in_folder if f.is_file()) / (1024 * 1024)
        
        folder_stats[folder_name] = (file_count, total_size)
        
        # Formatowanie nazwy folderu
        folder_display = folder_path.name
        
        if file_count > 0:
            print(f"  📁 {folder_display:20s} - {file_count:2d} plików ({total_size:6.2f} MB)")
            
            # Pokaż pierwsze 2 pliki
            for f in sorted(files_in_folder)[:2]:
                if f.is_file():
                    print(f"     ├─ {f.name}")
            if file_count > 2:
                print(f"     └─ ... i {file_count - 2} więcej")
        else:
            print(f"  📁 {folder_display:20s} - (pusty)")

print("\n" + "="*70)
total_files = sum(f[0] for f in folder_stats.values())
total_size = sum(f[1] for f in folder_stats.values())
print(f"✅ RAZEM: {total_files} plików | {total_size:.2f} MB")
print("="*70)

print("\n🎯 GLAVNY PLIK DO ODESŁANIA:")
main_file_path = SUBFOLDERS['main_pdf'] / "LISTA3_WSZYSTKIE_WYKRESY.pdf"
if main_file_path.exists():
    print(f"   ✓ {main_file_path.name}")
    print(f"   ✓ Ścieżka: PDF_do_odesłania/LISTA3_WSZYSTKIE_WYKRESY.pdf")
    print(f"   ✓ Zawiera wszystkie wykresy (zadania 1-21)")
    print(f"   ✓ Każdy wykres: tytuł + osie + legenda")
else:
    print("   ⚠️  Plik jeszcze się tworzy...")

print("\n" + "="*70)


📊 PODSUMOWANIE - STRUKTURA PLIKÓW

✅ WSZYSTKIE 21 ZADAŃ UKOŃCZONE!

Struktura folderów:

  📁 PDF_indywidualne     - 32 plików (  3.24 MB)
     ├─ 01_ekstrema.pdf
     ├─ 02_trzy_funkcje.pdf
     └─ ... i 30 więcej
  📁 PNG                  - 32 plików (  7.27 MB)
     ├─ 01_ekstrema.png
     ├─ 02_trzy_funkcje.png
     └─ ... i 30 więcej
  📁 JPG                  - 32 plików (  2.96 MB)
     ├─ 01_ekstrema.jpg
     ├─ 02_trzy_funkcje.jpg
     └─ ... i 30 więcej
  📁 PDF_do_odesłania     -  1 plików (  3.24 MB)
     ├─ LISTA3_WSZYSTKIE_WYKRESY.pdf

✅ RAZEM: 97 plików | 16.71 MB

🎯 GLAVNY PLIK DO ODESŁANIA:
   ✓ LISTA3_WSZYSTKIE_WYKRESY.pdf
   ✓ Ścieżka: PDF_do_odesłania/LISTA3_WSZYSTKIE_WYKRESY.pdf
   ✓ Zawiera wszystkie wykresy (zadania 1-21)
   ✓ Każdy wykres: tytuł + osie + legenda

